In [1]:
using LinearAlgebra, Polynomials, Plots
using Revise, DelimitedFiles, BenchmarkTools
using CloudAtlas, BifurcationKit, ChannelflowWrapper
using Dates
using Random
using Base.Threads

"""
    myreaddlm(filename, cc='%')

Read matrix or vector from a file, dropping comments marked with cc.
"""
function myreaddlm(filename; cc='%')
    X = readdlm(filename, comments=true, comment_char=cc)
    if size(X,2) == 1
        X = X[:,1]
    end
    X
end

macro suppress(ex)
    quote
        # Generate a unique name for the old stdout to avoid variable collision
        local old_stdout = stdout
        redirect_stdout(devnull)
        try
            # We use esc(ex) to run the expression in the caller's scope
            $(esc(ex))
        finally
            redirect_stdout(old_stdout)
        end
    end
end

sx, sy, sz, tx, tz = halfbox_symmetries()

pwd()

"/home/ebenq/Dev/julia/CloudAtlas.jl/notebooks/tw_fuzzing"

In [2]:
# Parameters
hookparams = SearchParams(ftol=1e-08, xtol=1e-12, Nnewton=30,Nhook=8,δ=0.01, verbosity=0)
Re = 200
cx0 = 0.000 # values from paper
cz0 = 0.009

α, γ = 1.0, 2.0                     # Fourier wavenumbers α, γ = 2π/Lx, 2π/Lz
H = [(sx*sy)*(tx*tz)]               # Generators of the symmetric subspace of TW1
normalize = true                    # Normalize the basis set or not?

# The DNS file to project from (ensure this path is correct)
dns_file = "TW1-2pi1piRe200-40x49x40.nc" 

# List of resolutions to test: [(J, K, L), ...]
# discretizations = [(1, 1, 1), (1, 1, 2), (1, 1, 3), (1, 2, 3), (1, 3, 5), (2, 4, 7), (3, 5, 9)]
discretizations = [(1, 1, 3), (1, 2, 3), (1, 3, 5), (2, 4, 7)]
# discretizations = [(1, 1, 3), (1, 2, 3)]

4-element Vector{Tuple{Int64, Int64, Int64}}:
 (1, 1, 3)
 (1, 2, 3)
 (1, 3, 5)
 (2, 4, 7)

In [3]:
"""
    fuzz_symmetry_space(discretizations, H, Re; attempts_per_level=10)

1. Loops through discretizations (J,K,L).
2. Generates random spectral guesses.
3. Attempts to converge using the low-dim ODE solver (`hookstepsolve`).
4. If the ODE solver converges, reconstructs the field and runs `findsoln`.
"""
function fuzz_symmetry_space(discretizations, H, Re; 
                             attempts_per_level=25, 
                             base_dir="fuzz_results",
                             noise_scale=1e-2,
                             symm_file = "./sxytxz.asc",
                             reference_path = "./TW1-2pi1piRe200-40x49x40.nc",
                             norm_threshold=1e-3,
                             xnorm = 0.40,
    )
    
    # 1. Setup Directory
    mkpath(base_dir)
    
    # --- Thread Safety Tools ---
    io_lock = ReentrantLock()        # Prevents jumbled print output
    total_found = Atomic{Int}(0)     # Thread-safe counter
    
    println("Starting Parallel Fuzz Search in $base_dir with $(nthreads()) threads")

    for (J, K, L) in discretizations
        # Use the lock to print cleanly
        lock(io_lock) do 
            println("\n" * "="^60)
            println("  Discretization: J=$J, K=$K, L=$L")
            println("="^60)
        end

        # Pre-calculate model for this level (shared by all threads)
        α, γ = 1.0, 2.0 
        model = TWModel(α, γ, J, K, L, H; normalize=false)
        m = length(model)

        @threads for i in 1:attempts_per_level
            
            # 1. Generate Random Guess (Thread-local)
            # x_guess = (randn(m) .* noise_scale)
            x_guess = randn(m)
            x_guess = xnorm/norm(x_guess) * x_guess
            cx_guess = randn() * 0.1
            cz_guess = randn() * 0.1
            ξ_guess = [x_guess; cx_guess; cz_guess]

            # 2. Try Low-Dimensional Solve
            # Note: We create new closures inside the loop so they are thread-safe
            f(ξ) = model.g(ξ, Re)
            Df(ξ) = model.Dg(ξ, Re)
            params = SearchParams(ftol=1e-6, xtol=1e-6, Nnewton=15, Nhook=4, verbosity=0)
            
            # Suppress output per thread to keep terminal clean
            ξ_star, converged = hookstepsolve(f, Df, ξ_guess, params)

            # Check convergence AND non-triviality
            solution_norm = norm(ξ_star[1:m])
            
            if converged && solution_norm > norm_threshold
                # Atomic add: safely increment counter
                atomic_add!(total_found, 1)
                
                # Lock output: safely print success message
                lock(io_lock) do
                    println("  [Thread $(threadid())] Hit! Converged at attempt #$i")
                end
                
                # 3. Promote to findsoln
                # Use 'i' in folder name to ensure unique paths
                timestamp = Dates.format(now(), "MM-DD-HHMMSS")
                sol_dir = joinpath(base_dir, "sol_$(J)_$(K)_$(L)_id$(i)_$(timestamp)")
                mkpath(sol_dir)
                
                guess_path = joinpath(sol_dir, "u_guess.nc")
                
                # We lock file generation just to be safe with disk I/O bursts
                lock(io_lock) do
                    coeff2field(ξ_star[1:m], model.ijkl, reference_path, guess_path)
                end
                
                try
                    # Run findsoln (External Binary)
                    # Note: Each thread launches its own process. 
                    findsoln(guess_path;
                        R = Re, eqb = true, xrel = model.keep_cx, zrel = model.keep_cz,
                        symms = abspath(symm_file), od = sol_dir,
                        T = 20
                    )
                catch e
                    lock(io_lock) do
                        println("  [Thread $(threadid())] findsoln failed: $e")
                    end
                end
            end
        end
    end
end

fuzz_symmetry_space

In [4]:
fuzz_symmetry_space(discretizations, H, 200.0; attempts_per_level=25)

Starting Parallel Fuzz Search in fuzz_results with 1 threads

  Discretization: J=1, K=1, L=3
J,K,L,m == 1,1,3,33
(2J+1)(2K+1)(2L+1) + 1 == 64
Making matrices B, A1, A2, Cx, Cz...
Phase constraints: keep_cx = false, keep_cz = true
Making quadratic operator N...
1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 

  Discretization: J=1, K=2, L=3
J,K,L,m == 1,2,3,53
(2J+1)(2K+1)(2L+1) + 1 == 106
Making matrices B, A1, A2, Cx, Cz...
Phase constraints: keep_cx = false, keep_cz = true
Making quadratic operator N...
1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 
  [Thread 1] Hit! Converged at attempt #12
alpha, gamma == 1, 2.00000000000006
Nx, Ny, Nz == 40, 49, 40
Reading ijkl indices of basis set from file
reading N == 53 ijkl indices
ijkl[0] == 1 0 0 1
L == max l == 3
Constructing Legendre polynomials
Assigning Polynomial, size = 1 
Assigning Polyno